### GCB - RULE

In [ ]:
import pdfplumber
import pandas as pd
import re

pdf_path = r"C:\Users\GIDI\Desktop\Kwaku\1769004698273.pdf"
output_excel = "GCB_statement_FINAL.xlsx"

def to_float(x):
    if not x: return 0.0
    return float(x.replace("GHS", "").replace(",", "").strip())

# --- TUNED COORDINATE ZONES ---
DATE_MAX_X = 80        
DESC_START_X = 110     
REF_START_X = 280           # Lowered to 280 to catch/remove Reference numbers from Description
CREDIT_DEBIT_X_LIMIT = 400  # <--- CRITICAL FIX: Split line between 350 (Credit) and 440 (Debit)

records = []
current_txn = None 

with pdfplumber.open(pdf_path) as pdf:
    for page in pdf.pages:
        words = page.extract_words(use_text_flow=True)

        rows = {}
        for w in words:
            y = round(w["top"] / 2) * 2 
            rows.setdefault(y, []).append(w)

        sorted_row_keys = sorted(rows.keys())

        for y in sorted_row_keys:
            row_words = sorted(rows[y], key=lambda x: x["x0"])
            
            row_date_parts = []
            row_desc_parts = []
            row_amounts = []

            for w in row_words:
                text = w["text"]
                x = w["x0"]

                # 1. Date Zone
                if x < DATE_MAX_X:
                    if (text.isdigit() and len(text) <= 2) or \
                       (text.isalpha() and len(text) == 3) or \
                       (text.isdigit() and len(text) == 4):
                         row_date_parts.append(text)
                    continue

                # 2. Amount Zone
                if "GHS" in text:
                    row_amounts.append((x, text))
                    continue

                # 3. Reference Zone (Ignore these words)
                if x >= REF_START_X and x < CREDIT_DEBIT_X_LIMIT - 10:
                     continue 

                # 4. Description Zone
                if x >= DESC_START_X:
                    # Simple dedup: don't add if it's identical to the last word (fixes bold text issues)
                    if not row_desc_parts or text != row_desc_parts[-1]:
                        row_desc_parts.append(text)

            # --- ROW PROCESSING ---
            
            # CASE A: NEW TRANSACTION
            if len(row_date_parts) == 3 and row_amounts:
                if current_txn:
                    records.append(current_txn)

                value_date = " ".join(row_date_parts)
                description = " ".join(row_desc_parts)

                row_amounts = sorted(row_amounts, key=lambda x: x[0])
                
                # Balance is always the last amount
                balance_x, balance_txt = row_amounts[-1]
                balance = to_float(balance_txt)
                
                credit = debit = None
                
                if len(row_amounts) >= 2:
                    txn_x, txn_txt = row_amounts[-2]
                    txn_val = to_float(txn_txt)
                    
                    # Split based on X=400
                    if txn_x < CREDIT_DEBIT_X_LIMIT:
                        credit = txn_val  # Left side (approx 350)
                    else:
                        debit = txn_val   # Right side (approx 440)

                current_txn = {
                    "Value Date": value_date,
                    "Description": description,
                    "Credit": credit,
                    "Debit": debit,
                    "Balance": balance
                }

            # CASE B: CONTINUATION
            elif not row_date_parts and not row_amounts and current_txn:
                if row_desc_parts:
                    # Add space only if not already there
                    new_text = " ".join(row_desc_parts)
                    if new_text not in current_txn["Description"]:
                        current_txn["Description"] += " " + new_text

if current_txn:
    records.append(current_txn)

# --- EXPORT ---
df = pd.DataFrame(records)
df['Description'] = df['Description'].str.strip()
df = df.fillna('')

df = df.iloc[::-1]

df.to_excel(output_excel, index=False)
print(f"✅ Excel created with {len(df)} rows.")
print(df[['Value Date', 'Description', 'Debit', 'Credit']].head().to_string())

✅ Excel created with 121 rows.
    Value Date                                                                    Description    Debit Credit
0  19 Jan 2026           Clg Chq pd-GRA CUSTO 555INWC260190485 MS DIVISION TOR~000099 /000099  2306.06       
1  15 Jan 2026  260115132909571||GRA| 101EPAY260150368 |TASM260115000503 (Kyeremeh Hutchison)   2850.0       
2  15 Jan 2026  260115130835150||GRA| 101EPAY260150367 |TASM260115000433 (Kyeremeh Hutchison)  19421.2       
3  13 Jan 2026                       206012026010800151-OI 101EPAY260130356 LCORP ENERGIA LTD  12244.5       
4  13 Jan 2026                                         MOMO AND GIP TRANSFER 101GIPO260130068     10.0       


### GCB - ACCOUNTING

In [38]:
import pdfplumber
import pandas as pd

pdf_path = r"C:\Users\GIDI\Desktop\Kwaku\1769004698273.pdf"
output_excel = "GCB_statement_FINAL.xlsx"

def to_float(x):
    if not x: return 0.0
    return float(x.replace("GHS", "").replace(",", "").strip())

records = []

# --- CONFIGURATION ---
DATE_MAX_X = 80       
DESC_START_X = 110    
REF_START_X = 280     

with pdfplumber.open(pdf_path) as pdf:
    for page in pdf.pages:
        words = page.extract_words(use_text_flow=True)
        
        # Group by Row (Y-axis)
        rows = {}
        for w in words:
            y = round(w["top"] / 2) * 2
            rows.setdefault(y, []).append(w)
        
        sorted_rows = sorted(rows.keys())
        
        last_txn_index = -1

        for y in sorted_rows:
            row_words = sorted(rows[y], key=lambda x: x["x0"])
            
            row_date = []
            row_desc = []
            row_amounts = [] 

            for w in row_words:
                text = w["text"]
                x = w["x0"]

                # 1. Capture Date
                if x < DATE_MAX_X:
                    if (text.isdigit() and len(text) <= 2) or \
                       (text.isalpha() and len(text) == 3) or \
                       (text.isdigit() and len(text) == 4):
                         row_date.append(text)
                    continue

                # 2. Capture Amounts
                if "GHS" in text:
                    row_amounts.append((x, text))
                    continue

                # 3. Skip Reference Number Zone
                if x >= REF_START_X and x < 400: 
                     continue 

                # 4. Capture Description
                if x >= DESC_START_X:
                    if not row_desc or text != row_desc[-1]:
                        row_desc.append(text)

            # --- PROCESS ROW ---
            if len(row_date) == 3 and row_amounts:
                # NEW TRANSACTION FOUND
                value_date = " ".join(row_date)
                description = " ".join(row_desc)
                
                # Sort amounts left-to-right
                row_amounts = sorted(row_amounts, key=lambda x: x[0])
                
                # Last amount is Balance
                balance = to_float(row_amounts[-1][1])
                
                # Second-to-last is Transaction Amount
                txn_amount = 0.0
                txn_x = 0
                if len(row_amounts) >= 2:
                    txn_x = row_amounts[-2][0]
                    txn_amount = to_float(row_amounts[-2][1])

                # Store raw data (Credit/Debit determined later)
                records.append({
                    "Value Date": value_date,
                    "Description": description,
                    "Raw_Amount": txn_amount,
                    "Balance": balance,
                    "Txn_X": txn_x,
                    "Debit": None,
                    "Credit": None
                })
                last_txn_index = len(records) - 1

            # --- CONTINUATION LINES ---
            elif not row_date and not row_amounts and last_txn_index >= 0:
                if row_desc:
                    records[last_txn_index]["Description"] += " " + " ".join(row_desc)

# --- APPLY ACCOUNTING LOGIC (Credit vs Debit) ---
for i in range(len(records)):
    current_txn = records[i]
    current_bal = current_txn["Balance"]
    amount = current_txn["Raw_Amount"]
    
    if amount == 0:
        continue

    # Compare with the NEXT row (since statements are usually newest-first)
    if i < len(records) - 1:
        prev_row_bal = records[i+1]["Balance"]
        diff = current_bal - prev_row_bal
        
        # Balance Increased = Credit
        if diff > 0: 
            current_txn["Credit"] = amount
        # Balance Decreased = Debit
        else:
            current_txn["Debit"] = amount
            
    # Fallback for the very last row (use position average)
    else:
        all_debit_xs = [r["Txn_X"] for r in records if r["Debit"] is not None]
        avg_debit_x = sum(all_debit_xs) / len(all_debit_xs) if all_debit_xs else 450
        
        if current_txn["Txn_X"] > avg_debit_x - 20: 
            current_txn["Debit"] = amount
        else:
            current_txn["Credit"] = amount

# --- EXPORT AND REORDER ---
df = pd.DataFrame(records)

# 1. Fill NaN with empty string
df['Credit'] = df['Credit'].fillna('')
df['Debit'] = df['Debit'].fillna('')

# 2. Select Columns
df = df[['Value Date', 'Description', 'Credit', 'Debit', 'Balance']]

# 3. Clean whitespace
df['Description'] = df['Description'].str.strip()

# 4. REVERSE THE ORDER (Last to First)
df = df.iloc[::-1]  # <--- THIS FLIPS THE TABLE

df.to_excel(output_excel, index=False)
print(f"✅ Excel created with {len(df)} rows (Reversed Order).")
print(df.head().to_string())

✅ Excel created with 121 rows (Reversed Order).
      Value Date                                                                                                                                                   Description     Credit     Debit     Balance
120  01 Dec 2025                                                                                                                 IB Spirits Petroleum 133FTIN253350002 limited    95850.0             446549.19
119  01 Dec 2025        300304******0000 000 137GIPI253350PW0 00001 OILCORPENERGIALTD KYEREMEH YEREMEH OILCORP ENERGIA LIMITED ERGIA LIMITED FTR-Oilcorp interaccount-15332224    40000.0             486549.19
118  01 Dec 2025                                                                               Cheque Withdrawal BY 164CQWL253350068 HUTCHINSON AMPAABENG KYEREMEH /000089 MEH             220000.0   266549.19
117  02 Dec 2025                                                                                                        